In [1]:
# mike babb
# created: 2026 08 23
# find five groups of five letters

In [2]:
# standard
import math
import pickle
import os
import sqlite3
from string import ascii_lowercase
from itertools import product, permutations, combinations

In [3]:
import pandas as pd
import numpy as np

In [4]:
# custom
from utils import *
import _run_constants as rc

# LOAD DATA

In [5]:
word_df, word_id_list, word_byte_list, word_byte_array, word_byte_to_word_dict = load_input_data()

In [6]:
# confirm that these words - a valid solution - are present
outcome_words = ['vibex', 'glyph', 'muntz', 'dwarf', 'jocks']
for ow in outcome_words:
    print(ow, ''.join(sorted(ow)), ''.join(sorted(ow)) in word_df['letters_sorted'].tolist())

vibex beivx True
glyph ghlpy True
muntz mntuz True
dwarf adfrw True
jocks cjkos True


## EXAMPLES OF BYTE COMPARISONS

In [7]:
w1 = 'abhor'
w2 = 'cleft'
w3 = 'frown'
w1b = byte_encode_words(w1)
w2b = byte_encode_words(w2)
w3b = byte_encode_words(w3)

In [8]:
# no letters in common
w1b & w2b

0

In [9]:
# letters in common
w1b & w3b

147456

In [10]:
# bitwise or
w1b | w2b

673975

In [11]:
# this is the same as directly above
byte_encode_words('abhorcleft')

673975

In [12]:
byte_encode_words(ascii_lowercase)

67108863

# BUILD LEVEL 2

In [13]:
l2_list = np.full(shape = (10000000, 3), fill_value = -1, dtype = np.int32)
row_index = 0
found_values = set()
for w1_be, w2_be in combinations(word_byte_list, 2):
    if w1_be & w2_be == 0:   
        # they share no letters in common
        l2 = w1_be | w2_be                          
        
        l2_list[row_index, :] = np.array([w1_be, w2_be, l2], dtype = np.int32)
        found_values.add(l2)
        row_index += 1

# trim the data frame
l2_list = l2_list[:row_index, :]
print(l2_list.shape)
l2_df = pd.DataFrame(data = l2_list, columns = ['w1b', 'w2b', 'l2'])

(3213696, 3)


In [14]:
l2_df['l2'].unique().shape

(640023,)

In [19]:
curr_l2_list = l2_df['l2'].unique()

In [24]:
value_splits = list(range(0, curr_l2_list.shape[0] + 100, 100))

In [25]:
len(value_splits)

6402

In [30]:
output = curr_l2_list[0] | curr_l2_list

In [28]:
start_pos = 0
output_list = np.zeros(shape = (10000000, 3), dtype = np.int32)
for i_v, v in enumerate(value_splits[1:]):
    prev_v = value_splits[i_v]


    combinations = np.array(np.meshgrid(curr_l2_list[prev_v:v], curr_l2_list)).T.reshape(-1, 2)
    positional_idx_l2l3l4 = (combinations[:, 0] & combinations[:, 1]) == 0
    if positional_idx_l2l3l4.sum() > 0:

        # l2 words with different letters
        combinations = combinations[positional_idx_l2l3l4]    
            
        # l2, l3, l4 accumulated letters
        output_array_l2l3l4 = combinations[:, 0] | combinations[:, 1]
        n_rows = output_array_l2l3l4.shape[0]
        
            
        temp_output = np.zeros(shape = (n_rows, 3), dtype = np.int32)
        temp_output[:, 0] = combinations[:, 0]
        temp_output[:, 1] = combinations[:, 1]
        temp_output[:, 2] = output_array_l2l3l4
        output_list[start_pos:start_pos + n_rows, :] = temp_output
        start_pos += n_rows
    

    if i_v % 10 == 0:
        print(i_v)

0
10
20
30
40
50


KeyboardInterrupt: 

In [ ]:
# BUILD THE CHAR MATRIX AND THE SELECTOR

In [ ]:
l2_df.head()

In [ ]:
l2_df['w1'] = l2_df['w1b'].map(word_byte_to_word_dict)
l2_df['w2'] = l2_df['w2b'].map(word_byte_to_word_dict)

In [ ]:
l2_df['l2_words'] = l2_df['w1'] + l2_df['w2']

In [ ]:
l2_df.head()

In [ ]:
letter_dict = {l:p for p,l in enumerate(ascii_lowercase)}

In [ ]:
char_matrix = np.zeros(shape = (l2_df.shape[0], 26), dtype = np.int8)
for i_l2, l2 in enumerate(l2_df['l2_words']):    
    for l in l2:
        char_matrix[i_l2, letter_dict[l]] += 1

    if i_l2 % 10000 == 0:
        print(i_l2)

In [ ]:
# what are the indices of words that do no have the letter b?
for l, p in letter_dict.items():
    idx = char_matrix[:, p] == 0
    curr_l2_list = l2_list[idx, :]
    unique_curr_l2_list = np.unique(curr_l2_list[:, 2])
    print(l, curr_l2_list.shape, unique_curr_l2_list.shape)


In [ ]:
# just the letter b
idx = char_matrix[:, 1] == 0
curr_l2_list = l2_list[idx, :]
curr_char_matrix = char_matrix[idx, :]
unique_curr_l2_list = np.unique(curr_l2_list[:, 2])
print('b', curr_l2_list.shape, unique_curr_l2_list.shape)


In [ ]:
curr_char_matrix.shape

In [ ]:
# how do we dramatically winnow down the search space?
# it feels like I can come up with a way to group each set of letters and then add them. 
# which is what I am already doing. 
# but it's comparison after comparison. 

In [ ]:
# Define two arrays
array1 = np.array([1, 2, 3])
array2 = np.array([4, 5])

# Generate combinations using meshgrid and reshape
combinations = np.array(np.meshgrid(array1, array2)).T.reshape(-1, 2)
print(combinations)

In [ ]:
curr_l2_list.shape

In [ ]:
unique_curr_l2_list.shape

In [ ]:
# split up the list

In [ ]:
l2_df.shape

In [ ]:
test_l2_df = l2_df.drop_duplicates(subset = 'l2').reset_index(drop = True)

In [ ]:
test_l2_df.shape

In [ ]:
curr_l2_list = test_l2_df['l2'].to_numpy(dtype = np.int32)

In [ ]:
value_splits = list(range(0, curr_l2_list.shape[0] + 10, 10))

In [ ]:
# this right here is the output
# it's just not faster to split things up because it causes too much enumeration
# the letter selector situation, all over again

In [ ]:
curr_l2_list.shape

In [ ]:
test_l2_df = l2_df.loc[idx, :].reset_index(drop = True)

In [ ]:
test_l2_df.head()

In [ ]:
test_l2_df['l2'].unique().shape

In [ ]:
test_l2_df.shape

In [ ]:
mike = byte_encode_words('mike')
mike

In [ ]:
babb = byte_encode_words('babb')
babb

In [ ]:
bike = byte_encode_words('bike')
bike

In [ ]:
mike & babb

In [ ]:
mike | babb

In [ ]:
mike & bike

In [ ]:
mike | bike

In [ ]:
babb | bike

In [ ]:
working_test_l2_df = test_l2_df.drop_duplicates(subset = 'l2').reset_index()

In [ ]:
output_list = np.zeros(shape = (10000000, 2), dtype = np.int32)
test_l2_all = working_test_l2_df['l2'].to_numpy(dtype = np.int32)
start_pos = 0
for i_l2, l2 in np.ndenumerate(test_l2_all):
    # compare the current l2 to all l2 - this will find all instances
    # indexer for l2, l3, and l4
    positional_idx_l2l3l4 = (test_l2_all & l2) == 0

    # l2 words with different letters
    output_array_w3bw4b = test_l2_all[positional_idx_l2l3l4]    
    
    # l2, l3, l4 accumulated letters
    output_array_l2l3l4 = output_array_w3bw4b | l2
    n_rows = output_array_l2l3l4.shape[0]

    if n_rows > 0:
        temp_output = np.zeros(shape = (n_rows, 2), dtype = np.int32)
        temp_output[:, 0] = l2
        temp_output[:, 1] = output_array_l2l3l4
        output_list[start_pos:start_pos + n_rows, :] = temp_output
        start_pos += n_rows

    if i_l2[0] % 10000 == 0:
        print(i_l2)


    



# BUILD LEVELS 3 THROUGH 5

In [ ]:
l2_df.shape

In [ ]:
l2_all = l2_list[:, 2]

In [ ]:
l2_all.shape

In [32]:
l2_df_test = l2_df.drop_duplicates(subset = 'l2').reset_index(drop = True)

In [33]:
l2_df_test.shape

(640023, 3)

In [49]:
l2_all = l2_df_test['l2'].to_numpy(dtype=np.int32)

In [51]:
l2_all

array([  284959,   553247,   808991, ..., 51962115, 55723281, 60179473],
      shape=(640023,), dtype=int32)

In [52]:
l2_all[0]

np.int32(284959)

In [66]:
output = (l2_all & l2_all[10]) == 0
output.sum()

np.int64(4)

In [67]:
output.any()

np.True_

In [68]:
# so, now, let's try computing all possible pairs
start_pos = 0
total_output = np.full(shape = (100000000, 5), fill_value= -1, dtype = np.int32)
row_index = 0
for i_row, row in l2_df_test.iterrows():    
    # levels 1 and 2
    w1b, w2b, l2 = row

    # compare the current l2 to all l2 - this will find all instances
    # indexer for l2, l3, and l4
    positional_idx_l2l3l4 = (l2_all & l2) == 0
    if positional_idx_l2l3l4.any() > 0:


        # combined l2 words and words with different letters
        output_array_w3bw4b = l2_all[positional_idx_l2l3l4]    
        
        # l2, l3, l4 accumulated letters
        output_array_l2l3l4 = output_array_w3bw4b | l2

        # create the temp output
        n_rows_l2l3l4 = output_array_l2l3l4.shape[0]    

        # check against the word_byte_arry for w5b
        
        for w3bw4b, l2l3l4 in zip(output_array_w3bw4b, output_array_l2l3l4):

            positional_idx_l2l3l4l5 = (l2l3l4 & word_byte_array) == 0
            if positional_idx_l2l3l4l5.any() > 0: 
                output_array_l2l3l4l5 = word_byte_array[positional_idx_l2l3l4l5]
                n_rows_l2l3l4l5 = output_array_l2l3l4l5.shape[0]
            
                temp_output = np.zeros(shape = (n_rows_l2l3l4l5, 5), dtype = np.int32)
                temp_output[:, 0] = w1b
                temp_output[:, 1] = w2b

                # calcualte w3b and w4b
                # this will look up the 'other' w1b and w2b values
                #w3bw4b = l2_df_test.loc[l2_df_test['l2'] == l2, ['w1b', 'w2b']].to_numpy(dtype = np.int32)                
                temp_output[:, 2] = w3bw4b
        
                # w3b
                #temp_output[:, 2] = w3bw4b[:, 0]
        
                #w4b
                #temp_output[:, 3] = w3bw4b[:, 1]

                #w5b
                temp_output[:, 4] = output_array_l2l3l4l5
        
                total_output[start_pos:start_pos + n_rows_l2l3l4l5, :] = temp_output
        
                start_pos += n_rows_l2l3l4l5        
    
    if i_row % 1000 == 0:
        # print the number of l2 iterations and the shape of the output
        print(i_row, row_index)

    row_index += 1

0 0
1000 1000
2000 2000
3000 3000
4000 4000
5000 5000
6000 6000
7000 7000
8000 8000
9000 9000
10000 10000
11000 11000
12000 12000
13000 13000
14000 14000
15000 15000
16000 16000
17000 17000
18000 18000
19000 19000
20000 20000
21000 21000
22000 22000
23000 23000
24000 24000
25000 25000
26000 26000
27000 27000
28000 28000
29000 29000
30000 30000
31000 31000
32000 32000
33000 33000
34000 34000
35000 35000
36000 36000
37000 37000
38000 38000
39000 39000
40000 40000
41000 41000
42000 42000
43000 43000
44000 44000
45000 45000
46000 46000
47000 47000
48000 48000
49000 49000
50000 50000
51000 51000
52000 52000
53000 53000
54000 54000
55000 55000
56000 56000
57000 57000
58000 58000
59000 59000
60000 60000
61000 61000
62000 62000
63000 63000
64000 64000
65000 65000
66000 66000
67000 67000
68000 68000
69000 69000
70000 70000
71000 71000
72000 72000
73000 73000
74000 74000
75000 75000
76000 76000
77000 77000
78000 78000
79000 79000
80000 80000
81000 81000
82000 82000
83000 83000
84000 84000
85000 

In [69]:
testo = total_output[:start_pos]

In [70]:
testo.shape

(11008, 5)

In [ ]:
testo[0, 0] | testo[0, 1] | testo[0, 2] | testo [0, 3]

In [ ]:
np.save('testo.npy', arr = testo)

In [ ]:
testo

In [ ]:
grand_output.shape

In [ ]:
grand_output

In [ ]:
go_df = pd.DataFrame(data = grand_output, columns = ['l2l3l4', 'w5b'])

In [ ]:
testo_unique.shape

In [ ]:
w4_df = pd.DataFrame(data = testo_unique, columns = ['w1b', 'w2b', 'w3b', 'w4b', 'l2', 'l3l4', 'l2l3l4'])

In [ ]:
w4_df.shape

In [ ]:
outcome = pd.merge(left = w4_df, right = go_df)

In [ ]:
outcome.shape

In [ ]:
# add the words
for cn_idx in range(1, 6):
    b_cn = f"w{cn_idx}b"
    w_cn = f"w{cn_idx}"
    outcome[w_cn] = outcome[b_cn].map(word_byte_to_word_dict)

In [ ]:
outcome.head()

In [ ]:
# sort the words
col_names = ['w1', 'w2', 'w3', 'w4', 'w5']
outcome['word_group'] = outcome[col_names].apply(lambda x: ''.join(sorted(x)), axis = 1)

In [ ]:
outcome.head()

In [ ]:
# drop duplicates across the word fields
col_names = ['w1', 'w2', 'w3', 'w4', 'w5', 'word_group']
o_word_df = outcome[col_names].drop_duplicates(subset='word_group')

In [ ]:
o_word_df.shape

In [ ]:
o_word_df.head()

In [ ]:
# so, now, let's try computing all possible pairs
total_output = np.full(shape = (1000000, 9), fill_value= -1, dtype = np.int32)
row_index = 0
for i_row, row in l2_df.iterrows():    
    # levels 1 and 2
    w1b, w2b, l2 = row

    # indexer for l3
    positional_idx_l3 = (word_byte_array & l2) == 0

    # l3 words with different letters
    output_array_w3b = word_byte_array[positional_idx_l3]    

    # l3 accumulated letters
    output_array_l3 = output_array_w3b | l2

    ## enumerate level 3
    for w3b, l3 in zip(output_array_w3b, output_array_l3):

        # build level 4

        # l4 idx
        positional_idx_l4 = (word_byte_array & l3) == 0

        # words with different letters
        output_array_w4b = word_byte_array[positional_idx_l4]    
        
        # accumulated letters
        output_array_l4 = output_array_w4b | l3

        ## enumerate level 5
        for w4b, l4 in zip(output_array_w4b, output_array_l4):

            # build level 5

            # l5 idx
            positional_idx_l5 = (word_byte_array & l4) == 0
            
            # words with different letters
            output_array_w5b = word_byte_array[positional_idx_l5]    

            if output_array_w5b.size > 0:
                    
                # accumulated letters
                output_array_l5 = output_array_w5b | l4

                ## gather and combine the output
                for w5b, l5 in zip(output_array_w5b, output_array_l5):

                    temp_list = np.array([w1b, w2b, w3b, w4b, w5b, l2, l3, l4, l5], dtype = np.int32)
                    total_output[row_index, :] = temp_list                   
                    row_index += 1


    if i_row % 10000 == 0:
        # print the number of l2 iterations and the shape of the output
        print(i_row, row_index)
    


# CREATE AND SAVE OUTPUT

In [ ]:
total_output = total_output[:row_index, :]
col_names = ['w1b', 'w2b', 'w3b', 'w4b', 'w5b', 'l2', 'l3', 'l4', 'l5']
l5_df = pd.DataFrame(data = total_output, columns = col_names)


In [ ]:
l5_df.shape

In [ ]:
l5_df.head()

In [ ]:
l5_df.tail()

In [ ]:
l5_df.to_csv(path_or_buf='l5.txt', sep = '\t', index = False)
